In [2]:
import os
import sys
from io import StringIO
import pickle

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from openpyxl import load_workbook

In [5]:
# Helper functions to handle messy and poorly defined data structures.

def preprocess_csv(path):
    with open(path, "r") as f:
        lines = f.readlines()
    # Process the lines as needed
    lines = [line.split(",") for line in lines]
    length = len(lines[0])
    lines = [line[:length] for line in lines]
    lines = [",".join(line) for line in lines]
    return lines


def preprocess_dat(path):
    with open(path, "r") as f:
        lines = f.readlines()

    # First two characters of the directory name encode the year
    year = path.split("/")[-1][:2]
    special_case = 0

    # In 1995 they started to change the format.
    # They did not succeed in 1995. 
    # In 1996 they only changed the extension, handled elsewhere.
    if int(year) == 95:
        special_case = 1
    
    # In 1984 they first defined the current format.
    # Thus, the format is not yet fully defined, and is changed for later years.
    # But I can't complain because I refuse to work with whatever was been done the years prior.
    elif int(year) == 84:
        special_case = 2

    # One day python will have switch
    if special_case == 0:
        header = ["year", "state", "Car Type", "manufacturer_id", "Model Name", "CID_value", "Fuel Sys", "Transmission", "CYL_value", "YR", "Model ID", "unk_1", "unk_2", "unk_model_company_id", "unk_model_id", "Manufacturer", "EST_CTY", "EST_HWY", "EST_CMB", "ADJ_CTY", "ADJ_HWY", "ADJ_CMB", "Fuel Cost", "CID Type", "CYL Type", "Passanger", "Trunk", "Cargo", "O/D", "Drive System", "Fuel Type", "Transmission Description"]
        lines = [line.ljust(250) for line in lines]
        lines = [[line[:4], line[4], line[5:7], line[7:10], line[10:39], line[39:42], line[42:44], line[44:50], line[50:52], line[52:54], line[54:66], line[66:68], line[68:70], line[70:73], line[73:83], line[83:117], line[117:119], line[122:124], line[127:129], line[132:134], line[137:139], line[142:144], line[147:152], line[152:160], line[160:190], line[190:200], line[200:210], line[210:220], line[220:224], line[224:227], line[229], line[230:250]] for line in lines]
        lines = [[item.strip() for item in line] for line in lines]
        liness = [header]
        liness.extend(lines)
        lines = liness
        lines = [";".join(line) for line in lines]

    elif special_case == 1:
        header = ["year", "state", "Car Type", "manufacturer_id", "Model Name", "CID_value", "Fuel Sys", "Transmission", "CYL_value", "YR", "Model ID", "unk_1", "unk_2", "unk_model_company_id", "unk_model_id", "Manufacturer", "EST_CTY", "EST_HWY", "EST_CMB", "ADJ_CTY", "ADJ_HWY", "ADJ_CMB", "Fuel Cost", "CID Type", "CYL Type", "Passanger", "Trunk", "Cargo", "O/D", "Drive System", "Fuel Type", "Transmission Description"]
        lines = [line.ljust(250) for line in lines]
        lines = [line[12:] for line in lines]
        lines = [[line[:4], line[4], line[5:7], line[7:10], line[10:39], line[39:42], line[42:44], line[44:50], line[50:52], line[52:54], line[54:66], line[66:68], line[68:70], line[70:73], line[73:83], line[83:117], line[117:119], line[122:124], line[127:129], line[132:134], line[137:139], line[142:144], line[147:152], line[152:160], line[160:190], line[190:200], line[200:210], line[210:220], line[220:224], line[224:227], line[229], line[230:250]] for line in lines]
        lines = [[item.strip() for item in line] for line in lines]
        liness = [header]
        liness.extend(lines)
        lines = liness
        lines = [";".join(line) for line in lines]

    elif special_case == 2:
        header = ["year", "state", "Car Type", "manufacturer_id", "Model Name", "CID_value", "Fuel Sys", "Transmission", "CYL_value", "YR", "Model ID", "unk_1", "unk_2", "unk_model_company_id", "unk_model_id", "Manufacturer", "EST", "HWY", "CMB", "Fuel Cost", "CID Type", "CYL Type", "Passanger", "Trunk", "Cargo"]
        lines = [line.ljust(220) for line in lines]
        lines = [[line[:4], line[4], line[5:7], line[7:10], line[10:39], line[39:42], line[42:44], line[44:50], line[50:52], line[52:54], line[54:66], line[66:68], line[68:70], line[70:73], line[73:83], line[83:132], line[132:134], line[137:139], line[142:144], line[147:152], line[152:160], line[160:190], line[190:200], line[200:210], line[210:220]] for line in lines]
        lines = [[item.strip() for item in line] for line in lines]
        liness = [header]
        liness.extend(lines)
        lines = liness
        lines = [";".join(line) for line in lines]

    return lines

In [6]:
# Function in charge of loading the data from supported formats.

def load_data(path):
    # Ignore non-data directories
    try:
        format = path.split(".")[-1]
    except IndexError:
        raise ValueError("Invalid file path")
    
    if format == "csv": # Checked
        # Fixes some rows having more columns than defined categories on header
        lines = preprocess_csv(path) 
        data = pd.read_csv(StringIO("\n".join(lines)))

    elif format == "xls": # Checked
        # No issue detected on load. Care is needed on which file to load.
        data = pd.read_excel(path) 
    elif format == "xlsx": # Checked
        # Assume first visible sheet is the database
        wb = load_workbook(path)
        visible_sheets = [ws for ws in wb.worksheets if ws.sheet_state == "visible"]
        sheet = visible_sheets[0]
        data = pd.DataFrame(sheet.values, columns=next(sheet.values))
        data = data[1:]  

    elif format == "DAT": # Checked
        # Fixes arbitrary and archaic definition of data structure
        lines = preprocess_dat(path) 
        data = pd.read_csv(StringIO("\n".join(lines)), sep=";", index_col=False)
    
    elif format == "ASC": 
        # Included as DAT because it only changes the extension
        lines = preprocess_dat(path)
        data = pd.read_csv(StringIO("\n".join(lines)), sep=";", index_col=False)
    
    else:
        raise ValueError("Unsupported file format")
    
    # Purge virtually empty columns
    data = data.loc[:, data.notna().sum() >= 5]

    return data

In [7]:
# Define a dictionary with the dataframes per year.

def combine_data(folders, structured = True):
    dataframes = {}
    for folder in folders:
        yr = folder[:2]
        files = os.listdir(f"data/{folder}")
        if yr.isnumeric():
            if len(files) > 0:
                dataframes[yr] = {}
                for file in files:
                    try:
                        data = load_data(f"data/{folder}/{file}")
                        dataframes[yr][file.split(".")[-1]] = data
                    except:
                        continue
    if structured:
        return structure_data(dataframes)
    else:
        return dataframes
    

# Structure the previous dictionary for easier access and to avoid redundancy.

def structure_data(dataframes):
    # If more than one file exists for a year, follow priority order
    data_dictionary = {}
    priority = ["xlsx", "csv", "xls", "DAT", "ASC"]
    
    for year, formats in dataframes.items():
        
        chosen_value = None
        chosen_format = None
        for fmt in priority:
            if fmt in formats:
                chosen_value = formats[fmt]
                chosen_format = fmt
                break


        data_dictionary[year] = {"format": chosen_format, "data": chosen_value, "header": list(chosen_value.columns) if chosen_value is not None else None}

    return data_dictionary

In [ ]:
folders = os.listdir("data")
dataframes = combine_data(folders, structured=True)

In [ ]:
# Pickle df
with open("data/dataframes.pkl", "wb") as f:
    pickle.dump(dataframes, f)

In [ ]:
for year, vals in dataframes.items():
    print(year, vals["header"])

00 ['Class', 'Manufacturer', 'carline name', 'displ', 'cyl', 'trans', 'drv', 'cty', 'hwy', 'cmb', 'ucty', 'uhwy', 'ucmb', 'fl', 'G', 'T', 'S', '2pv', '2lv', '4pv', '4lv', 'hpv', 'hlv', 'fcost', 'eng dscr', 'trans dscr', 'vpc', 'cls']
01 ['Class', 'Manufacturer', 'carline name', 'displ', 'cyl', 'trans', 'drv', 'cty', 'hwy', 'cmb', 'ucty', 'uhwy', 'ucmb', 'fl', 'G', 'T', 'S', '2pv', '2lv', '4pv', '4lv', 'hpv', 'hlv', 'fcost', 'eng dscr', 'trans dscr', 'vpc', 'cls']
02 ['Class', 'Manufacturer', 'carline name', 'displ', 'cyl', 'trans', 'drv', 'bidx', 'cty', 'hwy', 'cmb', 'ucty', 'uhwy', 'ucmb', 'fl', 'G', 'T', 'S', '2pv', '2lv', '4pv', '4lv', 'hpv', 'hlv', 'fcost', 'eng dscr', 'trans dscr', 'vpc', 'cls']
03 ['Class', 'Manufacturer', 'carline name', 'displ', 'cyl', 'trans', 'drv', 'bidx', 'cty', 'hwy', 'cmb', 'ucty', 'uhwy', 'ucmb', 'fl', 'G', 'T', 'S', '2pv', '2lv', '4pv', '4lv', 'hpv', 'hlv', 'fcost', 'eng dscr', 'trans dscr', 'vpc', 'cls']
04 ['Class', 'Manufacturer', 'carline name', 'di

Data are stored per year in different dataframes, all as pandas.DataFrame objects.

Next step is to combine them. For this to be possible, however, the headers need to be matched, and the data be adjusted if needed.

Timeframes with the same headers are:
1.  YR 84
2.  YR 85:97
3.  YR 98:99
4.  YR 00:01
5.  YR 02:05
6.  YR 06
7.  YR 07 (One extra column than 06: INDEX_NUMB)
8.  YR 08 (One extra column than 07: REL_DT)
9.  YR 09 (Kinda explodes in Unnamed columns)
10. YR 10
11. YR 11 (Very similar from 10, until it's not)
12. YR 12 (Okay it's just one space on the second column label)
13. YR 13 
14. YR 14:24 (Finally a standard)
15. YR 25:26 (Okay probably just an additional space)

In [4]:
with open("data/dataframes.pkl", "rb") as f:
    dataframes = pickle.load(f)

In [5]:
dataframes

{'00': {'format': 'xls',
  'data':                             Class Manufacturer            carline name  displ  \
  0                     TWO SEATERS        ACURA                     NSX    3.0   
  1                     TWO SEATERS        ACURA                     NSX    3.2   
  2                     TWO SEATERS          BMW                 M COUPE    3.2   
  3                     TWO SEATERS          BMW                Z3 COUPE    2.8   
  4                     TWO SEATERS          BMW                Z3 COUPE    2.8   
  ..                            ...          ...                     ...    ...   
  840  SPEC PURP VEH - S.U.V. - 4WD       TOYOTA  LAND CRUISER WAGON 4WD    4.7   
  841  SPEC PURP VEH - S.U.V. - 4WD       TOYOTA                RAV4 4WD    2.0   
  842  SPEC PURP VEH - S.U.V. - 4WD       TOYOTA                RAV4 4WD    2.0   
  843  SPEC PURP VEH - S.U.V. - 4WD       TOYOTA       RAV4 SOFT TOP 4WD    2.0   
  844  SPEC PURP VEH - S.U.V. - 4WD       TOYOTA      